# 08 Prepare WLASL2000 Deployment Package

## What this notebook does
This notebook copies the selected WLASL2000 model and label map into an app-ready folder.

## Main outputs
```text
app/models/ASL/WLASL2000/selected_wlasl2000_model.pt
app/models/ASL/WLASL2000/asl_wlasl2000_labels.json
app/models/ASL/WLASL2000/wlasl2000_deployment_config.json
```

## Why this matters
The app should load a clean model path, label map path, input shape, and confidence rules without depending on notebook variables.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import shutil

## 1. Set paths

In [ ]:
PROJECT_ROOT = Path("E:/Be_My_Ear")

DATASET_NAME = "WLASL2000"
PREFIX = "wlasl2000"

MODEL_DIR = PROJECT_ROOT / "models" / "ASL" / DATASET_NAME
REPORT_DIR = PROJECT_ROOT / "reports" / f"phase1_{PREFIX}"

LABEL_MAP_FILE = PROJECT_ROOT / "data" / "label_maps" / DATASET_NAME / f"asl_{PREFIX}_labels.json"

DEPLOY_DIR = PROJECT_ROOT / "app" / "models" / "ASL" / DATASET_NAME
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

COMPARISON_FILE = REPORT_DIR / f"{PREFIX}_selected_model_comparison.csv"

print("Comparison exists:", COMPARISON_FILE.exists(), COMPARISON_FILE)
print("Label map exists:", LABEL_MAP_FILE.exists(), LABEL_MAP_FILE)
print("Deploy folder:", DEPLOY_DIR)

## 2. Select best model from comparison

In [ ]:
comparison_df = pd.read_csv(COMPARISON_FILE)
comparison_df = comparison_df.sort_values("test_top1_accuracy", ascending=False)

selected = comparison_df.iloc[0].to_dict()

print("Selected model:", selected["model"])
print("Top-1:", selected["test_top1_accuracy"])
print("Top-3:", selected["test_top3_accuracy"])
print("Top-5:", selected["test_top5_accuracy"])
print("Macro F1:", selected["test_macro_f1"])
print("Model path:", selected["model_path"])

## 3. Copy model and label map

In [ ]:
source_model_path = Path(selected["model_path"])

target_model_path = DEPLOY_DIR / "selected_wlasl2000_model.pt"
target_label_map_path = DEPLOY_DIR / "asl_wlasl2000_labels.json"

shutil.copy2(source_model_path, target_model_path)
shutil.copy2(LABEL_MAP_FILE, target_label_map_path)

print("Copied model to:", target_model_path)
print("Copied label map to:", target_label_map_path)

## 4. Create deployment config

In [ ]:
deployment_config = {
    "project": "Be My Ear",
    "language": "ASL",
    "dataset": DATASET_NAME,
    "selected_model_name": selected["model"],
    "model_path": str(target_model_path),
    "label_map_path": str(target_label_map_path),
    "input_shape": [60, 516],
    "base_keypoint_shape": [60, 258],
    "features": "MediaPipe Holistic keypoints + velocity",
    "sequence_length": 60,
    "num_classes": int(selected["classes"]),
    "confidence_rules": {
        "speak_top1_threshold": 0.70,
        "show_top5_min_threshold": 0.40,
        "ask_to_repeat_below": 0.40
    },
    "metrics": {
        "test_top1_accuracy": float(selected["test_top1_accuracy"]),
        "test_top3_accuracy": float(selected["test_top3_accuracy"]),
        "test_top5_accuracy": float(selected["test_top5_accuracy"]),
        "test_macro_f1": float(selected["test_macro_f1"]),
        "best_val_f1": float(selected["best_val_f1"]),
        "best_val_top5": float(selected["best_val_top5"])
    },
    "deployment_notes": [
        "Use MediaPipe Holistic to extract 60-frame keypoint sequences.",
        "Concatenate keypoints with velocity to create input shape (60, 516).",
        "Do not always speak Top-1 directly.",
        "Use confidence threshold rules.",
        "For medium confidence, show Top-5 choices.",
        "For low confidence, ask the user to sign again."
    ]
}

deployment_config_file = DEPLOY_DIR / "wlasl2000_deployment_config.json"

with open(deployment_config_file, "w", encoding="utf-8") as f:
    json.dump(deployment_config, f, indent=4)

print("Saved deployment config:", deployment_config_file)

deployment_config

## 5. Final deployment summary

In [ ]:
print("WLASL2000 deployment package ready")
print("----------------------------------")
print("Model:", target_model_path)
print("Label map:", target_label_map_path)
print("Config:", deployment_config_file)
print()
print("App decision logic:")
print("confidence >= 0.70 → speak/show Top-1")
print("0.40–0.70 → show Top-5 suggestions")
print("confidence < 0.40 → ask user to sign again")